# Play two videos side by side, including over Remote SSH

Remote notebooks often fail to play videos when the browser cannot directly read the remote file path. This notebook provides two display modes:

1. **Embedded mode**: converts the video files to base64 data URLs inside the notebook output. This usually works in VS Code Remote-SSH and JupyterLab, but it can make the notebook/output large.
2. **Served-file mode**: uses relative file URLs. This is better for large videos, but requires the videos to be in the same folder as the notebook or a subfolder served by Jupyter.

For best browser compatibility, use MP4 with H.264 video and AAC audio.


In [ ]:
# Set these to paths on the REMOTE machine where the notebook kernel runs.
# Example: "./videos/left.mp4" or "/home/your_user/data/left.mp4"
video_left = "/home/yanghong/source/pyskl/output/demo_output_027_fft_seed0_yolo26l.mp4"
video_right = "/home/yanghong/source/pyskl/output/demo_output_027_fft_seed0_yolo26l_d2_ep10.mp4"

label_left = "YOLO26l-pose"
label_right = "YOLO26l-pose finetuned on keypoints dataset v2"

# Choose display mode:
# - True: most reliable over Remote-SSH / VS Code webviews, but large files make output large.
# - False: better for large files, but only works when browser/Jupyter can serve the files.
embed_videos = True


In [ ]:
from pathlib import Path
from IPython.display import HTML, display
from html import escape
import base64
import mimetypes
import os
import shutil


def _guess_mime(path_or_url):
    mime, _ = mimetypes.guess_type(str(path_or_url))
    return mime or "video/mp4"


def _file_to_data_url(path):
    path = Path(path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f"Video not found on the remote/kernel machine: {path}")
    mime = _guess_mime(path)
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{data}"


def _make_served_copy(path, target_dir="served_videos"):
    # Copy/symlink a video into a notebook-relative folder for served-file mode.
    src = Path(path).expanduser().resolve()
    if not src.exists():
        raise FileNotFoundError(f"Video not found on the remote/kernel machine: {src}")
    out_dir = Path(target_dir)
    out_dir.mkdir(exist_ok=True)
    dest = out_dir / src.name
    if not dest.exists():
        try:
            os.symlink(src, dest)
        except Exception:
            shutil.copy2(src, dest)
    return dest.as_posix()


def show_videos_side_by_side(left_src, right_src, left_label="Video 1", right_label="Video 2", embed=True, width="48%"):
    # Display two videos side by side with shared controls.
    if embed:
        left_url = _file_to_data_url(left_src)
        right_url = _file_to_data_url(right_src)
    else:
        left_url = _make_served_copy(left_src)
        right_url = _make_served_copy(right_src)

    html = f"""
    <div style="font-family: system-ui, sans-serif; max-width: 1200px;">
      <div style="display: flex; gap: 16px; align-items: flex-start; flex-wrap: wrap;">
        <div style="width: {escape(width)}; min-width: 320px;">
          <div style="font-weight: 600; margin-bottom: 6px;">{escape(left_label)}</div>
          <video id="vid_left" src="{left_url}" controls preload="metadata" style="width: 100%; border: 1px solid #ddd; border-radius: 8px;"></video>
        </div>
        <div style="width: {escape(width)}; min-width: 320px;">
          <div style="font-weight: 600; margin-bottom: 6px;">{escape(right_label)}</div>
          <video id="vid_right" src="{right_url}" controls preload="metadata" style="width: 100%; border: 1px solid #ddd; border-radius: 8px;"></video>
        </div>
      </div>
      <div style="margin-top: 12px; display: flex; gap: 8px; flex-wrap: wrap;">
        <button onclick="document.getElementById('vid_left').play(); document.getElementById('vid_right').play();">Play both</button>
        <button onclick="document.getElementById('vid_left').pause(); document.getElementById('vid_right').pause();">Pause both</button>
        <button onclick="document.getElementById('vid_left').currentTime=0; document.getElementById('vid_right').currentTime=0;">Reset both</button>
        <button onclick="document.getElementById('vid_right').currentTime = document.getElementById('vid_left').currentTime;">Sync right to left</button>
        <button onclick="document.getElementById('vid_left').currentTime = document.getElementById('vid_right').currentTime;">Sync left to right</button>
      </div>
    </div>
    """
    display(HTML(html))

show_videos_side_by_side(video_left, video_right, label_left, label_right, embed=embed_videos)


## Browser-compatible conversion

If a video appears but does not play, the issue is often the codec rather than the notebook path. Most browsers reliably play MP4 files encoded with H.264 video and AAC audio.

Run the following in a terminal on the remote machine, replacing filenames as needed:

```bash
ffmpeg -i input.mov -c:v libx264 -pix_fmt yuv420p -c:a aac output.mp4
```

Then set `video_left` or `video_right` to the converted `.mp4` file.


## Notes for Remote SSH

- `video_left` and `video_right` must point to files on the **remote machine**, not your local laptop.
- For VS Code Remote-SSH, `embed_videos = True` is usually the most reliable.
- For large videos, try `embed_videos = False`, but keep the videos in the notebook directory or let the notebook create the `served_videos/` folder.
- Very large embedded videos can make the notebook slow or create huge saved outputs. Clear the cell output after viewing if needed.
